# 03 — Gold: Dimensões SCD Tipo 1 (Spark SQL)

Processa 6 dimensões estáveis (SCD1) + DimDate usando **`MERGE INTO`** SQL.

**Técnica:** `spark.sql("MERGE INTO ...")` — upsert direto em SQL, sem DeltaTable API.

**Destaque:** DimEmployee — self-join + `collect_list` + `array_join` em SQL puro.

In [1]:
import sys
import os
sys.path.insert(0, os.getcwd())
from utils import get_spark, register_catalog, WAREHOUSE_DIR

spark = get_spark("NorthwindDW SQL - 03 Silver Dims")
print("Spark:", spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/03/29 00:45:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark: 3.5.0


In [2]:
register_catalog(spark)

Catálogo registrado: {'bronze': 11, 'silver': 0, 'gold': 11}


In [3]:
def scd1_merge(table_name, view_name, join_key):
    """MERGE INTO via Spark SQL — UPDATE se existe, INSERT se novo."""
    spark.sql(f"""
        MERGE INTO {table_name} AS tgt
        USING {view_name} AS src
        ON tgt.{join_key} = src.{join_key}
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)
    n = spark.sql(f"SELECT COUNT(*) AS n FROM {table_name}").collect()[0]["n"]
    print(f"  {table_name}: {n} linhas")
    return n

## DimEmployee — Dois problemas de modelagem em um

**Problema 1: hierarquia pai-filho**  
A tabela `Employees` tem `ReportsTo` apontando para outro `EmployeeID`.
Em um DW, a hierarquia deve ser achatada (1 coluna `ManagerName` na dim).
→ Técnica: self-join em bronze antes de popular gold.

**Problema 2: M:N Employee ↔ Territory**  
Cada empregado pode cobrir múltiplos territórios (tabela `EmployeeTerritories`).
Em uma dim SCD1, a abordagem mais simples é desnormalizar como lista textual.
→ Técnica: `collect_list()` + `array_join()` → `TerritoryList = "East, Midwest, West"`

*A solução M:N "correta" (Bridge Table) é demonstrada no notebook 10.*


In [4]:
# DimEmployee — SCD1 com hierarquia achatada + territórios
spark.sql("""
    CREATE OR REPLACE TEMP VIEW src_employee AS
    SELECT
        ABS(HASH(e.EmployeeID))                                       AS EmployeeSK,
        e.EmployeeID,
        CONCAT(e.FirstName, ' ', e.LastName)                          AS FullName,
        e.Title,
        CAST(e.HireDate AS DATE)                                       AS HireDate,
        e.City,
        e.Country,
        e.ReportsTo                                                    AS ReportsToID,
        CASE WHEN m.EmployeeID IS NOT NULL
             THEN CONCAT(m.FirstName, ' ', m.LastName)
        END                                                            AS ManagerName,
        COALESCE(ta.TerritoryList, '')                                AS TerritoryList,
        ta.RegionName,
        current_timestamp()                                            AS LoadTimestamp
    FROM bronze.employees e
    LEFT JOIN bronze.employees m ON e.ReportsTo = m.EmployeeID
    LEFT JOIN (
        SELECT
            et.EmployeeID,
            array_join(sort_array(collect_list(TRIM(t.TerritoryDescription))), ', ')
                AS TerritoryList,
            first(r.RegionDescription) AS RegionName
        FROM bronze.employee_territories et
        LEFT JOIN bronze.territories t  ON et.TerritoryID = t.TerritoryID
        LEFT JOIN bronze.region      r  ON t.RegionID     = r.RegionID
        GROUP BY et.EmployeeID
    ) ta ON e.EmployeeID = ta.EmployeeID
""")

print("DimEmployee — preview:")
spark.sql("""
    SELECT EmployeeID, FullName, ManagerName, TerritoryList FROM src_employee LIMIT 3
""").show(truncate=False)

scd1_merge("gold.DimEmployee", "src_employee", "EmployeeID")

DimEmployee — preview:


+----------+---------------+-------------+----------------------------------------------------------------------+
|EmployeeID|FullName       |ManagerName  |TerritoryList                                                         |
+----------+---------------+-------------+----------------------------------------------------------------------+
|2         |Andrew Fuller  |NULL         |Bedford, Boston, Braintree, Cambridge, Georgetow, Louisville, Westboro|
|1         |Nancy Davolio  |Andrew Fuller|Neward, Wilton                                                        |
|3         |Janet Leverling|Andrew Fuller|Atlanta, Orlando, Savannah, Tampa                                     |
+----------+---------------+-------------+----------------------------------------------------------------------+



  gold.DimEmployee: 9 linhas


9

In [5]:
# Resultado: 9 empregados com hierarquia achatada e lista de territórios
spark.sql("""
    SELECT FullName, ManagerName, TerritoryList, RegionName
    FROM gold.DimEmployee
""").show(truncate=False)


+----------------+---------------+------------------------------------------------------------------------------------------------------------------------------+--------------------------------------------------+
|FullName        |ManagerName    |TerritoryList                                                                                                                 |RegionName                                        |
+----------------+---------------+------------------------------------------------------------------------------------------------------------------------------+--------------------------------------------------+
|Michael Suyama  |Steven Buchanan|Bellevue, Phoenix, Redmond, Scottsdale, Seattle                                                                               |Western                                           |
|Anne Dodsworth  |Steven Buchanan|Bloomfield Hills, Hollis, Minneapolis, Portsmouth, Roseville, Southfield, Troy                                    

In [6]:
# DimCategory
spark.sql("""
    CREATE OR REPLACE TEMP VIEW src_category AS
    SELECT ABS(HASH(CategoryID)) AS CategorySK,
           CategoryID, CategoryName, Description, current_timestamp() AS LoadTimestamp
    FROM bronze.categories
""")
scd1_merge("gold.DimCategory", "src_category", "CategoryID")

  gold.DimCategory: 8 linhas


8

In [7]:
# DimSupplier
spark.sql("""
    CREATE OR REPLACE TEMP VIEW src_supplier AS
    SELECT ABS(HASH(SupplierID)) AS SupplierSK,
           SupplierID, CompanyName, City, Country, current_timestamp() AS LoadTimestamp
    FROM bronze.suppliers
""")
scd1_merge("gold.DimSupplier", "src_supplier", "SupplierID")

  gold.DimSupplier: 29 linhas


29

In [8]:
# DimShipper
spark.sql("""
    CREATE OR REPLACE TEMP VIEW src_shipper AS
    SELECT ABS(HASH(ShipperID)) AS ShipperSK,
           ShipperID, CompanyName, Phone, current_timestamp() AS LoadTimestamp
    FROM bronze.shippers
""")
scd1_merge("gold.DimShipper", "src_shipper", "ShipperID")

  gold.DimShipper: 3 linhas


3

In [9]:
# DimTerritory
spark.sql("""
    CREATE OR REPLACE TEMP VIEW src_territory AS
    SELECT ABS(HASH(t.TerritoryID)) AS TerritorySK,
           t.TerritoryID, t.TerritoryDescription, t.RegionID,
           r.RegionDescription AS RegionName,
           current_timestamp() AS LoadTimestamp
    FROM bronze.territories t
    LEFT JOIN bronze.region r ON t.RegionID = r.RegionID
""")
scd1_merge("gold.DimTerritory", "src_territory", "TerritoryID")

  gold.DimTerritory: 53 linhas


53

In [10]:
# DimDate — gerada via pandas (Spark não tem generate_series nativo)
import pandas as pd

existing = spark.sql("SELECT COUNT(*) AS n FROM gold.DimDate").collect()[0]["n"]
if existing > 0:
    print(f"DimDate já populada: {existing} dias. Pulando.")
else:
    dates = pd.date_range("1990-01-01", "2030-12-31", freq="D")
    pdf = pd.DataFrame({"FullDate": dates})
    pdf["DateKey"]   = pdf["FullDate"].dt.strftime("%Y%m%d").astype("int32")
    pdf["Year"]      = pdf["FullDate"].dt.year.astype("int32")
    pdf["Quarter"]   = pdf["FullDate"].dt.quarter.astype("int32")
    pdf["Month"]     = pdf["FullDate"].dt.month.astype("int32")
    pdf["MonthName"] = pdf["FullDate"].dt.strftime("%B")
    pdf["Day"]       = pdf["FullDate"].dt.day.astype("int32")
    pdf["DayOfWeek"] = pdf["FullDate"].dt.dayofweek.astype("int32")
    pdf["DayName"]   = pdf["FullDate"].dt.strftime("%A")
    pdf["IsWeekend"] = pdf["FullDate"].dt.dayofweek >= 5
    pdf["FullDate"]  = pdf["FullDate"].dt.date

    df_date = spark.createDataFrame(pdf)
    df_date.createOrReplaceTempView("src_dimdate")
    spark.sql("""
        INSERT INTO gold.DimDate
        SELECT DateKey, CAST(FullDate AS DATE), Year, Quarter, Month,
               MonthName, Day, DayOfWeek, DayName, IsWeekend
        FROM src_dimdate
    """)
    n = spark.sql("SELECT COUNT(*) AS n FROM gold.DimDate").collect()[0]["n"]
    print(f"DimDate populada: {n} dias (1990-01-01 a 2030-12-31)")

DimDate populada: 14975 dias (1990-01-01 a 2030-12-31)


In [11]:
silver_tables = [
    "gold.DimEmployee", "gold.DimCategory", "gold.DimSupplier",
    "gold.DimShipper", "gold.DimTerritory", "gold.DimDate",
]
print("\nContagens silver (SCD1 + DimDate):")
for t in silver_tables:
    n = spark.sql(f"SELECT COUNT(*) AS n FROM {t}").collect()[0]["n"]
    print(f"  {t:<35} {n:>6} linhas")

print("\nPreview DimEmployee (hierarquia):")
spark.sql("""
    SELECT FullName, Title, ManagerName, TerritoryList, RegionName
    FROM gold.DimEmployee
""").show(truncate=False)


Contagens silver (SCD1 + DimDate):


  gold.DimEmployee                         9 linhas


  gold.DimCategory                         8 linhas


  gold.DimSupplier                        29 linhas


  gold.DimShipper                          3 linhas


  gold.DimTerritory                       53 linhas


  gold.DimDate                         14975 linhas

Preview DimEmployee (hierarquia):


+----------------+------------------------+---------------+------------------------------------------------------------------------------------------------------------------------------+--------------------------------------------------+
|FullName        |Title                   |ManagerName    |TerritoryList                                                                                                                 |RegionName                                        |
+----------------+------------------------+---------------+------------------------------------------------------------------------------------------------------------------------------+--------------------------------------------------+
|Michael Suyama  |Sales Representative    |Steven Buchanan|Bellevue, Phoenix, Redmond, Scottsdale, Seattle                                                                               |Western                                           |
|Anne Dodsworth  |Sales Representative    |Steve